In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, auc
from sklearn.model_selection import train_test_split

from rapidgbm import RapidGBMTuner

import glob
import mplhep as hep
hep.style.use([hep.style.ATLAS])
import pickle

import uproot

In [2]:
def unpair_df(data):
    """
    Unpairs columns in the DataFrame that start with 'l1' or 'l2' by renaming them to 'l'
    and concatenating the resulting DataFrames vertically.

    Parameters:
    data (pd.DataFrame): The original DataFrame containing columns to be unpaired.

    Returns:
    pd.DataFrame: A new DataFrame with columns starting with 'l1' and 'l2' renamed to 'l',
                  and the DataFrame length doubled by concatenating the modified DataFrames.
    """
    l1_columns = [col for col in data.columns if 'l1' in col]
    l2_columns = [col for col in data.columns if 'l2' in col]
    other_columns = [col for col in data.columns if not ('l1' in col or 'l2' in col)]
    # Create a copy of the DataFrame with 'l1' and 'l2' columns renamed to 'l'
    data_l1 = data[other_columns + l1_columns].copy()
    data_l2 = data[other_columns + l2_columns].copy()
    data_l1.rename(columns={col: col.replace('l1', 'm_lx', 1) for col in l1_columns}, inplace=True)
    data_l2.rename(columns={col: col.replace('l2', 'm_lx', 1) for col in l2_columns}, inplace=True)
    # Concatenate the DataFrames vertically
    new_data = pd.concat([data_l1, data_l2], ignore_index=True)
    return new_data

def repair_df(data):
    """
    Pairs columns in the DataFrame that contain 'm_lx' by renaming them back to 'l1' and 'l2',
    and merges rows based on shared columns, resulting in the original DataFrame format.

    Parameters:
    data (pd.DataFrame): The DataFrame output from `unpair_df` function with doubled rows.

    Returns:
    pd.DataFrame: A DataFrame where columns containing 'm_lx' are renamed back to 'l1' and 'l2',
                  and rows are merged back to their original format with shared columns appearing only once.
    """
    # Split the data into two halves
    midpoint = len(data) // 2
    data_l1 = data.iloc[:midpoint].copy()
    data_l2 = data.iloc[midpoint:].copy()

    # Identify shared columns (not containing 'm_lx')
    shared_columns = [col for col in data.columns if 'm_lx' not in col]

    # Rename columns by replacing 'm_lx' with 'l1' in data_l1 and 'l2' in data_l2
    data_l1.rename(columns={col: col.replace('m_lx', 'l1') for col in data_l1.columns if 'm_lx' in col}, inplace=True)
    data_l2.rename(columns={col: col.replace('m_lx', 'l2') for col in data_l2.columns if 'm_lx' in col}, inplace=True)

    # Concatenate the two halves on columns, only including shared columns once
    paired_data = pd.concat(
        [data_l1.reset_index(drop=True)[shared_columns + [col for col in data_l1.columns if col not in shared_columns]],
         data_l2.reset_index(drop=True)[[col for col in data_l2.columns if col not in shared_columns]]],
        axis=1
    )

    return paired_data

def filter_columns(df, key_phrases):
    """
    Filters out columns that contain any of the key phrases in their names.

    Parameters:
    df (pd.DataFrame): The dataframe to filter.
    key_phrases (list): A list of key phrases to filter out.

    Returns:
    list: A list of column names that do not contain any of the key phrases.
    """
    filtered_columns = [col for col in df.columns if not any(phrase in col for phrase in key_phrases)]
    return filtered_columns


In [3]:
# # use glob to import all the files
# filePath = '/groups/hep/kinch/data/preprocessed_data/files_from_grid/user.amoranch.Zee_PhPy_601189_8Nov_hist/'

# # Use glob to search for all .root files recursively in the directory and its subdirectories
# root_files = glob.glob(f'{filePath}/**/*.root', recursive=True)

# size_zee = 0
# # Print the list of .root files
# for file in root_files:
#     zee_file = uproot.open(file)
#     zee_data = zee_file['HZG_Tree']
#     size_zee += len(zee_data["EventInfo.cutflow"].array())
#     print(len(zee_data["EventInfo.cutflow"].array()))
# print(f'number of saved zee-events: {size_zee}')

# filePath = '/groups/hep/kinch/data/preprocessed_data/files_from_grid/user.amoranch.ttbar_PhPy_601589_11Nov_hist/'

# root_files = glob.glob(f'{filePath}/**/*.root', recursive=True)

# size_ttbar = 0
# for file in root_files:
#     ttbar_file = uproot.open(file)
#     ttbar_data = ttbar_file['HZG_Tree']
#     size_ttbar += len(ttbar_data["EventInfo.cutflow"].array())
#     print(len(ttbar_data["EventInfo.cutflow"].array()))
# print(f'number of saved ttbar-events: {size_ttbar}')

In [4]:
filePath_zee = '/groups/hep/kinch/data/preprocessed_data/files_from_grid/user.amoranch.Zee_PhPy_601189_8Nov_hist/'
filePath_ttbar = '/groups/hep/kinch/data/preprocessed_data/files_from_grid/user.amoranch.ttbar_PhPy_601589_11Nov_hist/'
filePath_zmm = '/groups/hep/kinch/data/preprocessed_data/files_from_grid/user.amoranch.Zmm_PhPy_601190_11Nov_hist/'

In [5]:
root_files_zee = glob.glob(f'{filePath_zee}/**/*.root', recursive=True)
root_files_ttbar = glob.glob(f'{filePath_ttbar}/**/*.root', recursive=True)
root_files_zmm = glob.glob(f'{filePath_zmm}/**/*.root', recursive=True)


for file in root_files_zee:
    #check if .parquet file already exists
    if os.path.exists(f'{file[:-5]}.parquet'):
        continue
    zee_file = uproot.open(file)
    zee_data = zee_file['HZG_Tree']
    data_zee = pd.DataFrame.from_dict({key: zee_data[key].array() for key in zee_data.keys()})
    data_zee.to_parquet(f'{file[:-5]}.parquet')

for file in root_files_ttbar:
    if os.path.exists(f'{file[:-5]}.parquet'):
        continue
    ttbar_file = uproot.open(file)
    ttbar_data = ttbar_file['HZG_Tree']
    data_ttbar = pd.DataFrame.from_dict({key: ttbar_data[key].array() for key in ttbar_data.keys()})
    data_ttbar.to_parquet(f'{file[:-5]}.parquet')

for file in root_files_zmm:
    if os.path.exists(f'{file[:-5]}.parquet'):
        continue
    zmm_file = uproot.open(file)
    zmm_data = zmm_file['HZG_Tree']
    data_zmm = pd.DataFrame.from_dict({key: zmm_data[key].array() for key in zmm_data.keys()})
    data_zmm.to_parquet(f'{file[:-5]}.parquet')


In [18]:
iso_param_list = ['m_lx_ptcone20', 'm_lx_ptcone20_pt1000', 'm_lx_ptcone20_pt500', 'm_lx_ptcone30',
                  'm_lx_ptvarcone20', 'm_lx_ptvarcone30', 'm_lx_ptvarcone30_pt1000', 'm_lx_ptvarcone30_pt500',
                  'm_lx_topoetcone20', 'm_lx_topoetcone20ptCorrection', 'm_lx_topoetcone40', 'm_lx_neflowisol20']

iso_1_param_list = ['l1_ptcone20', 'l1_ptcone20_pt1000', 'l1_ptcone20_pt500', 'l1_ptcone30',
                    'l1_ptvarcone20', 'l1_ptvarcone30', 'l1_ptvarcone30_pt1000', 'l1_ptvarcone30_pt500',
                    'l1_topoetcone20', 'l1_topoetcone20ptCorrection', 'l1_topoetcone40', 'l1_neflowisol20']

iso_2_param_list = ['l2_ptcone20', 'l2_ptcone20_pt1000', 'l2_ptcone20_pt500', 'l2_ptcone30',
                    'l2_ptvarcone20', 'l2_ptvarcone30', 'l2_ptvarcone30_pt1000', 'l2_ptvarcone30_pt500',
                    'l2_topoetcone20', 'l2_topoetcone20ptCorrection', 'l2_topoetcone40', 'l2_neflowisol20']

pred_z_param_list = ['l1_iso_score', 'l2_iso_score', 
                     'l1_DNN_pel', 'l1_z0', 'l1_sig_z0', 'l2_DNN_pel', 'l2_z0', 'l2_sig_z0', 'EventInfo.actualIntPerXing',
                     'EventInfo.averageIntPerXing', 'EventInfo.mu', 'l1_DNN_pcf', 'l1_DNN_phf', 'l1_DNN_ple', 'l1_DNN_plh', 
                     'l1_DNN_ppc', 'l1_ECID', 'l1_charge', 'l2_DNN_pcf', 'l2_DNN_phf', 'l2_DNN_ple', 'l2_DNN_plh', 'l2_DNN_ppc',
                     'l2_ECID', 'l2_charge',]

others = ['EventInfo.eventNumber', 'truth_from_Z', 'EventInfo.cutflow', 'EventInfo.cutflow_new', 
          'll_m', 'l1_truthType', 'l2_truthType', 'llg_m', 'EventInfo.channel', 'l1_truthOrigin', 'l2_truthOrigin', 'l1_truthPdgId', 'l2_truthPdgId'
        ]

param_list = iso_1_param_list + iso_2_param_list + pred_z_param_list[2:] + others

In [13]:
parqeuet_files_zee = glob.glob(f'{filePath_zee}/**/*.parquet', recursive=True)
parqeuet_files_ttbar = glob.glob(f'{filePath_ttbar}/**/*.parquet', recursive=True)

In [21]:
test = pd.read_parquet(parqeuet_files_zee[0])
for col in test.columns:
    if 'l1' not in col:
        if col not in param_list:
            print(col)

Btag60_N_j
Btag70_N_j
Btag77_N_j
Btag85_N_j
Btag_N_j
Central_N_j
Central_pT_jj
EventInfo.PVx
EventInfo.PVy
EventInfo.PVz
MET_Dphi_ForwardJets
MET_Dphi_SoftJets
MET_Dphi_Zy
MET_Dphi_j1
MET_Dphi_l2
MET_Dphi_ll
MET_Dphi_ph
MET_RefEle
MET_RefGamma
MET_RefJets
MET_RefMuons
MET_met_PVSoftTrk
MET_met_SoftClus
MET_met_TST
MET_met_Truth
MET_phi_PVSoftTrk
MET_phi_TST
MET_sig_TST
MET_sig_Truth
MET_sumet_TST
MET_sumet_Truth
MET_x_TST
MET_y_TST
ML_l2_eta
ML_l2_m
ML_l2_phi
ML_l2_pt
ML_l3_eta
ML_l3_m
ML_l3_phi
ML_l3_pt
ML_l4_eta
ML_l4_m
ML_l4_phi
ML_l4_pt
VBF_BDTG
VBF_DRmin_y_j
VBF_Dphi_Zy_jj
VBF_Dphi_Zy_jj_FullRange
VBF_Dy_j_j
VBF_N_j
VBF_Zepp
VBF_eta_j1
VBF_eta_j2
VBF_m_jj
VBF_mass_j1
VBF_mass_j2
VBF_pT_j1
VBF_pT_j2
VBF_pT_jj
VBF_pTt_Zy
VBF_passFJVT_j1
VBF_passFJVT_j2
VBF_phi_j1
VBF_phi_j2
Z_truth_mass
Zy_Dphi_j1
etaTruthL0
etaTruthL1
etaTruthY
j1_eta_Truth
j1_m_Truth
j1_phi_Truth
j1_pt_Truth
j2_eta_Truth
j2_m_Truth
j2_phi_Truth
j2_pt_Truth
l2_CaloLRLikelihood
l2_CaloMuonIDTag
l2_DFCommonJetDr
l2_M

In [19]:

data_zee = pd.DataFrame()

for file in parqeuet_files_zee:
    new_file = pd.read_parquet(file)
    new_file = new_file[new_file['EventInfo.channel']==1]

    from_Z_bool = (new_file['l1_truthOrigin'] == 13) & (new_file['l2_truthOrigin'] == 13) & (new_file['l1_truthPdgId'] * new_file['l2_truthPdgId'] == -121)
    new_file['truth_from_Z'] = from_Z_bool.astype(bool)
    new_file = new_file[param_list]

    #iso_bool = (new_file['l1_truthType'] == 4) & (new_file['l2_truthType'] == 2) | (new_file['l1_truthType'] == 2) & (new_file['l2_truthType'] == 4)

    #new_file = new_file[~iso_bool]

    # new_file = unpair_df(new_file)

    data_zee = pd.concat([data_zee, new_file])
print(1)
data_ttbar = pd.DataFrame()
print(2)
for file in parqeuet_files_ttbar:
    new_file = pd.read_parquet(file)
    new_file = new_file[new_file['EventInfo.channel']==1]

    from_Z_bool = (new_file['l1_truthOrigin'] == 13) & (new_file['l2_truthOrigin'] == 13) & (new_file['l1_truthPdgId'] * new_file['l2_truthPdgId'] == -121)
    new_file['truth_from_Z'] = from_Z_bool.astype(bool)

    new_file = new_file[param_list]

    # new_file = unpair_df(new_file)

    # filter out double iso events

    iso_bool = (new_file['l1_truthType'] == 2) & (new_file['l2_truthType'] == 2) #| (new_file['l1_truthType'] == 4) & (new_file['l2_truthType'] == 2) | (new_file['l1_truthType'] == 2) & (new_file['l2_truthType'] == 4)

    new_file = new_file[~iso_bool]

    data_ttbar = pd.concat([data_ttbar, new_file])

1
2


In [19]:
print(data_zee[['l1_truthType', 'l2_truthType']].value_counts())

l1_truthType  l2_truthType
2.0           2.0             26744062
              17.0              516641
17.0          2.0                28099
4.0           17.0               23632
17.0          17.0               19300
4.0           4.0                12347
0.0           0.0                11467
2.0           0.0                 4765
              3.0                 3642
17.0          4.0                 3641
2.0           16.0                1335
0.0           16.0                 858
16.0          0.0                  813
2.0           15.0                 461
              7.0                  440
17.0          0.0                  395
0.0           17.0                 261
2.0           8.0                  231
4.0           0.0                  210
16.0          16.0                 194
3.0           2.0                  175
15.0          2.0                  145
4.0           3.0                  117
0.0           2.0                   83
3.0           17.0                  7

In [ ]:
# from_z_bool = (data_zee['l1_truthOrigin'] == 13) & (data_zee['l2_truthOrigin'] == 13) & (data_zee['l1_truthPdgId'] * data_zee['l2_truthPdgId'] == -121)

# data_zee['truth_from_Z'] = from_z_bool

# from_z_bool = (data_ttbar['l1_truthOrigin'] == 13) & (data_ttbar['l2_truthOrigin'] == 13) & (data_ttbar['l1_truthPdgId'] * data_ttbar['l2_truthPdgId'] == -121)
# print(from_z_bool.sum()/len(from_z_bool))

# data_ttbar['truth_from_Z'] = from_z_bool

In [20]:
data_combined = pd.concat([data_zee, data_ttbar])
data_fromZ = data_combined[data_combined['truth_from_Z'] == True]
data_not_fromZ = data_combined[data_combined['truth_from_Z'] == False]

print(data_fromZ.shape)
print(data_not_fromZ.shape)

if data_fromZ.shape[0] > data_not_fromZ.shape[0]:
    data_fromZ = data_fromZ.sample(data_not_fromZ.shape[0])
elif data_fromZ.shape[0] < data_not_fromZ.shape[0]:
    data_not_fromZ = data_not_fromZ.sample(data_fromZ.shape[0])

Data_BalZ = pd.concat([data_fromZ, data_not_fromZ])


Data_BalZ.to_parquet('/groups/hep/kinch/data/preprocessed_data/files_from_grid/Data_BalZ_ttbar_zee_ttbardoubleisofiltered.parquet')

fromZ = Data_BalZ['truth_from_Z'].values
print(np.sum(fromZ)/len(fromZ))

(26743963, 60)
(4198364, 60)
0.5


In [33]:
types_origins = pd.DataFrame()
for file in parqeuet_files_zee:
    new_file = pd.read_parquet(file)

    from_Z_bool = (new_file['l1_truthOrigin'] == 13) & (new_file['l2_truthOrigin'] == 13) & (new_file['l1_truthPdgId'] * new_file['l2_truthPdgId'] == -121)
    
    mask_of_annoying_decays = (new_file['l1_truthType'] == 2) & (new_file['l2_truthType'] == 2) | (new_file['l1_truthType'] == 4) & (new_file['l2_truthType'] == 2) | (new_file['l1_truthType'] == 2) & (new_file['l2_truthType'] == 4)

    small_sample = new_file[mask_of_annoying_decays & ~from_Z_bool][['l1_truthType', 'l2_truthType', 'l1_truthOrigin', 'l2_truthOrigin', 'l1_truthPdgId', 'l2_truthPdgId']]

    types_origins_zee = pd.concat([types_origins, small_sample])

types_origins_ttbar = pd.DataFrame()
for file in parqeuet_files_ttbar:
    new_file = pd.read_parquet(file)

    mask_of_annoying_decays = (new_file['l1_truthType'] == 2) & (new_file['l2_truthType'] == 2) | (new_file['l1_truthType'] == 4) & (new_file['l2_truthType'] == 2) | (new_file['l1_truthType'] == 2) & (new_file['l2_truthType'] == 4)

    from_Z_bool = (new_file['l1_truthOrigin'] == 13) & (new_file['l2_truthOrigin'] == 13) & (new_file['l1_truthPdgId'] * new_file['l2_truthPdgId'] == -121)

    small_sample = new_file[mask_of_annoying_decays & ~from_Z_bool][['l1_truthType', 'l2_truthType', 'l1_truthOrigin', 'l2_truthOrigin', 'l1_truthPdgId', 'l2_truthPdgId']]

    types_origins_ttbar = pd.concat([types_origins, small_sample])


In [27]:
print('origins ttbar from bck electron')
print(types_origins_ttbar[types_origins_ttbar['l1_truthType']==4]['l1_truthOrigin'].value_counts())
print(types_origins_ttbar[types_origins_ttbar['l2_truthType']==4]['l2_truthOrigin'].value_counts())

print('origins ttbar from isolated electron')
print(types_origins_ttbar[types_origins_ttbar['l1_truthType']==2]['l1_truthOrigin'].value_counts())
print(types_origins_ttbar[types_origins_ttbar['l2_truthType']==2]['l2_truthOrigin'].value_counts())

print('origins zee from bck electron')
print(types_origins_zee[types_origins_zee['l1_truthType']==4]['l1_truthOrigin'].value_counts())
print(types_origins_zee[types_origins_zee['l2_truthType']==4]['l2_truthOrigin'].value_counts())

print('origins zee from isolated electron')
print(types_origins_zee[types_origins_zee['l1_truthType']==2]['l1_truthOrigin'].value_counts())
print(types_origins_zee[types_origins_zee['l2_truthType']==2]['l2_truthOrigin'].value_counts())

origins ttbar from bck electron
l1_truthOrigin
5.0     949
6.0      51
27.0     11
23.0      5
Name: count, dtype: int64
l2_truthOrigin
5.0     6739
6.0      409
23.0      42
27.0      22
7.0        6
24.0       3
Name: count, dtype: int64
origins ttbar from isolated electron
l1_truthOrigin
10.0    22002
27.0        1
Name: count, dtype: int64
l2_truthOrigin
10.0    15796
27.0        2
Name: count, dtype: int64
origins zee from bck electron
l1_truthOrigin
5.0     55004
7.0        35
6.0        34
23.0        5
Name: count, dtype: int64
l2_truthOrigin
5.0     82371
6.0       445
7.0        72
23.0       51
27.0        7
Name: count, dtype: int64
origins zee from isolated electron
l1_truthOrigin
13.0    83133
Name: count, dtype: int64
l2_truthOrigin
13.0    55262
27.0        2
29.0        1
Name: count, dtype: int64


In [34]:
print(types_origins_zee[(types_origins_zee['l1_truthOrigin']==13) & (types_origins_zee['l2_truthOrigin']==13)][['l1_truthPdgId', 'l2_truthPdgId']].value_counts())

l1_truthPdgId  l2_truthPdgId
-1.0           -1.0             176
-11.0          -11.0              4
 11.0           11.0              4
Name: count, dtype: int64


In [ ]:
# print(f'number of events in Zee: {data_zee.shape[0]}, ratio of isolated electrons: {(len(data_zee[data_zee['l1_truthType']==2]) + len(data_zee[data_zee['l2_truthType']==2]))/(data_zee.shape[0]*2)}')
# print(f'number of events in ttbar: {data_ttbar.shape[0]}, ratio of isolated electrons: {(len(data_ttbar[data_ttbar['l1_truthType']==2]) + len(data_ttbar[data_ttbar['l2_truthType']==2]))/(data_ttbar.shape[0]*2)}')

# if data_zee.shape[0] > data_ttbar.shape[0]:
#     data_zee = data_zee.sample(n=data_ttbar.shape[0])
# else:
#     data_ttbar = data_ttbar.sample(n=data_zee.shape[0])

# Data_BalZ = pd.concat([data_zee, data_ttbar])
# Data_BalZ.to_parquet('Data_BalZ.parquet')

In [31]:
# data_zee = pd.concat([pd.read_parquet(file) for file in parqeuet_files_zee])
# data_ttbar = pd.concat([pd.read_parquet(file) for file in parqeuet_files_ttbar])

# from_z_bool = (data_zee['l1_truthOrigin'] == 13) & (data_zee['l2_truthOrigin'] == 13) & (data_zee['l1_truthPdgId'] * data_zee['l2_truthPdgId'] == -121)
# print(from_z_bool.sum()/len(from_z_bool))

# data_zee['truth_from_Z'] = from_z_bool

# from_z_bool = (data_ttbar['l1_truthOrigin'] == 13) & (data_ttbar['l2_truthOrigin'] == 13) & (data_ttbar['l1_truthPdgId'] * data_ttbar['l2_truthPdgId'] == -121)
# print(from_z_bool.sum()/len(from_z_bool))

# data_ttbar['truth_from_Z'] = from_z_bool

In [32]:
Data_unpaired = pd.concat([data_zee, data_ttbar])
# print(f'ratio of isolated electrons: {(len(Data[Data['l1_truthType']==2]) + len(Data[Data['l2_truthType']==2]))/(Data.shape[0]*2)}')

In [33]:
print(*Data_unpaired.columns)

EventInfo.actualIntPerXing EventInfo.averageIntPerXing EventInfo.mu EventInfo.eventNumber truth_from_Z EventInfo.cutflow EventInfo.cutflow_new ll_m m_lx_ptcone20 m_lx_ptcone20_pt1000 m_lx_ptcone20_pt500 m_lx_ptcone30 m_lx_ptvarcone20 m_lx_ptvarcone30 m_lx_ptvarcone30_pt1000 m_lx_ptvarcone30_pt500 m_lx_topoetcone20 m_lx_topoetcone20ptCorrection m_lx_topoetcone40 m_lx_neflowisol20 m_lx_DNN_pel m_lx_z0 m_lx_sig_z0 m_lx_truthType l1_neflowisom_lx0


In [34]:
# Data_unpaired_zee = unpair_df(data_zee)
# data_zee = None
print('got here')
# Data_unpaired_ttbar = unpair_df(data_ttbar)
# data_ttbar = None
print('got here')
data_zee = None
data_ttbar = None
print('got here')
Data_unpaired_iso = Data_unpaired[Data_unpaired['m_lx_truthType'] == 2]
Data_unpaired_noniso = Data_unpaired[Data_unpaired['m_lx_truthType'] != 2]

print(f'number of isolated electrons: {Data_unpaired_iso.shape[0]}, number of non-isolated electrons: {Data_unpaired_noniso.shape[0]}')

if Data_unpaired_iso.shape[0] > Data_unpaired_noniso.shape[0]:
    Data_unpaired_iso = Data_unpaired_iso.sample(n=Data_unpaired_noniso.shape[0])
elif Data_unpaired_iso.shape[0] < Data_unpaired_noniso.shape[0]:
    Data_unpaired_noniso = Data_unpaired_noniso.sample(n=Data_unpaired_iso.shape[0])

got here
got here
got here
number of isolated electrons: 56719363, number of non-isolated electrons: 5561065


In [35]:
# number of isolated electrons: 56731423, number of non-isolated electrons: 6789269

In [36]:
Data_BalIso = pd.concat([Data_unpaired_iso, Data_unpaired_noniso])

Data_BalIso.to_parquet('Data_BalIso.parquet')